Hi there!
This is the code template for CW1 task1 of COMP34711 2025/26.

- <span style="color:red; font-size:1em">First of all, please rename the notebook into "{your_student_id}_CW1_task1.ipynb", for example "12345678_CW1_task1.ipynb".</span>

- In this template, we only provide the minimal structure for your coursework.
  
- Please carefully read and organize your code in the template we provided.

## Constants

In [ ]:
#Please keep only necessary information in this cell.

#----------------------Please keep all following constants unchanged.----------------------------------------
NUM_ROWS_EXAMPLE = 100    # Number of rows in example.csv.
NUM_ROWS_TEST = 160    # Number of rows in testdata.csv.

#----------------------Please modify the following constants to fit your actual value.-----------------------
STUDENT_ID = 'your_student_id'  # Replace with your actual 8-digits student ID.
CORPUS = './data/WikiText_103.txt'  # Path to WikiText_103.txt
EXAMPLE_SET_INPUT = './data/CW1_examples.csv' # Path to CW1_examples.csv (no similarities)
EXAMPLE_SET_OUTPUT = f'./data/{STUDENT_ID}_CW1_examples_task1_results.csv'  # Path to your own prediction of CW1_examples.csv
EXAMPLE_SET_GOLD_STANDARD = './data/CW1_examples_gold.csv'  # Path to the CW1_examples_gold.csv
TEST_SET_INPUT = './data/CW1_testdata.csv' # Path to the CW1_testdata.csv

#----------------------Your constants--------------------------------------
# By adding more constants here, you can help improve the clarity and maintainability of your code and make the reviewing easier for TAs.

## Installations

In [ ]:
# Install required packages for the coursework

# Uncomment and run the following lines if packages are not installed
# !pip install pandas numpy


## Imports

In [ ]:
#Please keep all imports of your code cells in this cell

#---------------------Required imports----------------------
import pandas as pd
import re
import numpy as np
import math
import os.path
import sys
#----------------------Your imports-------------------------


## Start of your code cells

- The code cells provided below are demo code format for TAs to quickly locate your implementation.

- You have full right to freely add/delete/edit the titles and codes in the following cells.

### Corpus loader

In [ ]:
# Your code cell example

### Vectoriser



In [ ]:
# Your code cell example

### Out of vocabulary words processing

In [ ]:
# Your code cell example

### Multi-word terms processing

In [ ]:
# Your code cell example

### Similarity calculation

In [ ]:
# Your code cell example

## End of your code cells

In [ ]:
#@title Evaluation scripts

def read_data(submission_file_path, gold_standard_file_path):
    # Try to find your ID from the filename (looks for 7-8 digit numbers)
    id_regex = r'\d{7,8}'

    user_id = re.findall(id_regex, submission_file_path)
    print("Found your ID: ", user_id)
    if user_id:
        user_id = user_id[0]
    else:
        user_id = 'Unknown'

    # Load both CSV files (assuming no header row)
    submission_df = pd.read_csv(submission_file_path, header=None)
    reference_df = pd.read_csv(gold_standard_file_path, header=None)

    # Add row tracking to the reference data for precise comparisons
    indices = np.arange(reference_df.shape[0])
    reference_df['row_index'] = indices

    return submission_df, reference_df, user_id

def group_pairs_by_common_words(reference_df):
    """
    Group word pairs that share any common word (in either column 1 or column 2).
    """
    # Convert dataframe to list of dictionaries for easier processing
    all_pairs = reference_df.to_dict(orient='records')

    # Track which pairs have been assigned to groups
    assigned = [False] * len(all_pairs)
    groups = []

    for i, pair in enumerate(all_pairs):
        if assigned[i]:
            continue

        # Start a new group with this pair
        current_group = [pair]
        assigned[i] = True

        # Get words from this pair
        words_in_group = {pair[1], pair[2]}  # columns 1 and 2 are the word pair

        # Keep expanding the group by finding pairs that share words
        changed = True
        while changed:
            changed = False
            for j, other_pair in enumerate(all_pairs):
                if assigned[j]:
                    continue

                # Check if this pair shares any word with the current group
                other_words = {other_pair[1], other_pair[2]}
                if words_in_group & other_words:  # Set intersection - any common words
                    current_group.append(other_pair)
                    assigned[j] = True
                    words_in_group.update(other_words)
                    changed = True

        groups.append(current_group)

    return groups

def evaluate_submission(submission_df, reference_df, user_id):
    """
    Evaluate your submission by comparing similarity score orderings with the reference.

    1. Groups word pairs that share any common word (in either column 1 or column 2)
    2. Compares how you ranked similarity scores vs. the reference ranking
    3. Creates individual test cases to show exactly what passed or failed

    """
    # Group reference data by pairs that share any common word
    # This creates clusters of word pairs that have overlapping vocabulary
    grouped_list = group_pairs_by_common_words(reference_df)

    # Set up tracking for your evaluation results
    total_score = 0
    maximum_score = 0
    test_cases = []  # All individual test cases with their results
    failed_test_cases = []  # Failed tests for detailed feedback
    missed_rows = []  # Any data rows you might have missed
    test_case_id = 1

    # Process each group of word pairs
    for group_dict in grouped_list:
        # Skip single-item groups (no comparison possible)
        if len(group_dict) == 1:
            continue

        your_predictions = []

        # Find your predictions for each word pair in this group
        for row in group_dict:
            # Look up your prediction using the row ID (column 0)
            your_row = submission_df.loc[submission_df[0] == row[0]]

            try:
                your_row_dict = your_row.to_dict(orient='records')[0]
                your_predictions.append(your_row_dict)
            except:
                # Keep track of any missing data for helpful feedback
                missed_rows.append(row[0])
                continue

        # Sort both your predictions and reference data by row ID for fair comparison
        sorted_your_predictions = sorted(your_predictions, key=lambda x: x[0], reverse=True)
        sorted_reference_group = sorted(group_dict, key=lambda x: x[0], reverse=True)

        # Compare all possible pairs within this word group
        for i in range(len(sorted_your_predictions)):
            for j in range(i+1, len(sorted_your_predictions)):
                try:
                    # Calculate the difference in similarity scores (column 3)
                    your_result = float(sorted_your_predictions[i][3]) - float(sorted_your_predictions[j][3])
                    if math.isnan(your_result):
                        continue

                    reference_result = sorted_reference_group[i][3] - sorted_reference_group[j][3]

                    # Create a clear description for this test case
                    # Find common words between the two pairs being compared
                    words_pair1 = {sorted_reference_group[i][1], sorted_reference_group[i][2]}
                    words_pair2 = {sorted_reference_group[j][1], sorted_reference_group[j][2]}
                    common_words = words_pair1 & words_pair2
                    if common_words:
                        word_context = f"pairs sharing word(s): {', '.join(sorted(common_words))}"
                    else:
                        word_context = "connected word pairs"

                    pair1 = f"({sorted_reference_group[i][1]}, {sorted_reference_group[i][2]})"
                    pair2 = f"({sorted_reference_group[j][1]}, {sorted_reference_group[j][2]})"

                    # Figure out what the correct ordering should be
                    if reference_result > 0:
                        expected_order = f"{pair1} > {pair2}"
                    elif reference_result < 0:
                        expected_order = f"{pair1} < {pair2}"
                    else:
                        expected_order = f"{pair1} = {pair2}"

                    # And what your prediction actually was
                    if your_result > 0:
                        your_order = f"{pair1} > {pair2}"
                    elif your_result < 0:
                        your_order = f"{pair1} < {pair2}"
                    else:
                        your_order = f"{pair1} = {pair2}"

                    # Check if you got this comparison right!
                    test_passed = False
                    if your_result * reference_result > 0:
                        test_passed = True
                        total_score += 1
                    elif your_result == 0 and reference_result == 0:
                        test_passed = True
                        total_score += 1

                    # Record this test case for your feedback report
                    test_case = {
                        'id': test_case_id,
                        'description': f"Ordering comparison for {word_context}",
                        'expected': expected_order,
                        'actual': your_order,
                        'passed': test_passed,
                        'reference_data': [sorted_reference_group[i], sorted_reference_group[j]],
                        'your_data': [sorted_your_predictions[i], sorted_your_predictions[j]]
                    }

                    test_cases.append(test_case)

                    if not test_passed:
                        failed_test_cases.append(test_case)

                    test_case_id += 1

                except:
                    continue

                # Keep track of the total possible points
                maximum_score += 1

    return {
        'user_id': user_id,
        'total_score': total_score,
        'maximum_score': maximum_score,
        'test_cases': test_cases,
        'failed_test_cases': failed_test_cases,
        'missed_rows': missed_rows
    }

def create_progress_bar(passed, total, width=50):
    """
    Create a visual progress bar showing test results.

    Args:
        passed (int): Number of passed tests
        total (int): Total number of tests
        width (int): Width of the progress bar in characters

    Returns:
        str: Formatted progress bar string
    """
    if total == 0:
        return "[" + " " * width + "] 0/0 (0.0%)"

    percentage = (passed / total) * 100
    filled_width = int((passed / total) * width)
    failed_width = width - filled_width

    bar = "["
    bar += "✓" * filled_width  # Green checkmarks for passed tests
    bar += "✗" * failed_width  # Red X's for failed tests
    bar += f"] {passed}/{total} ({percentage:.1f}%)"

    return bar

def generate_feedback_report(result_dict):
    user_id = result_dict['user_id']
    total_score = result_dict['total_score']
    maximum_score = result_dict['maximum_score']
    test_cases = result_dict['test_cases']
    failed_test_cases = result_dict['failed_test_cases']
    missed_rows = result_dict['missed_rows']

    print("="*70)
    print("YOUR SUBMISSION EVALUATION REPORT")
    print("="*70)

    # Alert if ID not found in filename
    if user_id == 'Unknown':
        print('WARNING: ID not found in filename!')
        print('Please ensure your filename contains your 7-8 digit ID.')

    print(f"Your ID: {user_id}")
    print(f"Total Test Cases: {len(test_cases)}")
    print(f"Passed: {total_score}")
    print(f"Failed: {len(failed_test_cases)}")

    if maximum_score > 0:
        percentage = (total_score / maximum_score) * 100
        print(f"Success Rate: {percentage:.1f}%")
    else:
        print("Success Rate: N/A (no valid comparisons found)")


    # Show progress bar
    print("TEST RESULTS OVERVIEW:")
    progress_bar = create_progress_bar(total_score, maximum_score, 50)
    print(f"  {progress_bar}")
    print()

    # Generate detailed feedback for failed test cases
    if failed_test_cases:
        print(f"\nDETAILED FEEDBACK ON FAILED TEST CASES ({len(failed_test_cases)} failures):")
        print("=" * 70)

        for i, test_case in enumerate(failed_test_cases, 1):
            reference_data = test_case['reference_data']
            your_data = test_case['your_data']

            print(f"\nFAILED TEST CASE #{test_case['id']} ({i}/{len(failed_test_cases)})")
            print(f"Context: {test_case['description']}")
            print(f"Expected: {test_case['expected']}")
            print(f"Your Result: {test_case['actual']}")
            print("Detailed Comparison:")
            print("Reference (Correct):")

            # Show correct ordering from reference with actual scores
            for j in range(len(reference_data)):
                print(f"     {{{reference_data[j][0]}, {reference_data[j][1]}, {reference_data[j][2]}, {reference_data[j][3]}}}")

            print("Your Submission:")

            # Show your ordering with actual scores
            for j in range(len(your_data)):
                print(f"     {{{your_data[j][0]}, {your_data[j][1]}, {your_data[j][2]}, {your_data[j][3]}}}")

            print("-" * 50)
    else:
        print("\nEXCELLENT! All test cases passed!")
        print("Your similarity score orderings are completely correct!")

    # Report missing rows
    if missed_rows:
        print(f"\nMISSING DATA ({len(missed_rows)} rows not found):")
        print("-" * 40)
        for row in missed_rows:
            print(f"Row ID: {row} is missing from your submission")
        print("\nTIP: Make sure your submission includes all required rows.")
    else:
        print("\nDATA COMPLETENESS: No missing rows in your submission!")

    print("\n" + "="*70)

def mark_and_feedback(submission_file, reference_file):
    """
    Main function to run your submission evaluation script.

    Usage: python cw1_fast_evaluation.py <your_submission.csv> <reference_file.csv>
    """

    # Check if files exist
    if not os.path.exists(submission_file):
        print(f"Error: Your submission file '{submission_file}' not found!")
        print("Make sure the file path is correct and the file exists.")

    if not os.path.exists(reference_file):
        print(f"Error: Reference file '{reference_file}' not found!")
        print("Make sure you have the correct gold standard file.")

    try:
        print("Loading your data...")
        submission_df, reference_df, user_id = read_data(submission_file, reference_file)

        print("Evaluating your submission...")
        result_dict = evaluate_submission(submission_df, reference_df, user_id)

        print("Generating your personalized feedback report...")
        generate_feedback_report(result_dict)

    except Exception as e:
        print(f"Error during evaluation: {str(e)}")
        print("Please check that your files are in the correct CSV format.")
        print("Each row should contain: ID, word1, word2, similarity_score")

In [ ]:
# @title Evaluate the model on the example dataset

# Please run the evaluation scripts cell above before running the mark_and_record

results = mark_and_feedback(EXAMPLE_SET_OUTPUT, EXAMPLE_SET_GOLD_STANDARD)


### Save predictions to formatted file.

In [ ]:
# Now please modify the code to format your output csv file.

output_df = None  # Replace with your actual DataFrame or output
# For example, if you have a DataFrame named 'output_df', you can save it
assert isinstance(output_df, pd.DataFrame)
assert len(output_df) == NUM_ROWS_TEST, "Output length is not aligned with the testdata.csv."
output_df.to_csv(f'{STUDENT_ID}_CW1_task1_results.csv', index=False)